# IC-1109 — Case study: logistic regression or LDA for the credit committee?

**ML & FinTech · 115-1 · 20261109 · Session 3 · work in pairs · 40 minutes, then discussion**

| Step | Min | What you do |
|---|---|---|
| 0 | 3 | Sit with a partner. Fill in both IDs |
| 1 | 10 | An LDA classification **by hand** from two class means |
| 2 | 12 | Your desk: fit both models; compare accuracy |
| 3 | 10 | Count the applicants the two models treat differently; compare with your partner |
| 4 | 5 | Hand in **one sheet of paper per pair** |

You may use any AI tool for the code. The graded part is the paper your pair hands in, and what
you say when called on. Your numbers come from your 學號; your partner's differ.

---

## The case

The credit committee has to pick one default model for next year. The data team reports:

> *"Logistic regression and LDA give the **same accuracy** on our test set, to the second decimal.
> They are the same model in practice. Pick either."*

Your manager asks: **"Are they really the same? Would the same people be rejected?"**

## Setup

In [ ]:
import hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import warnings; warnings.filterwarnings("ignore")

In [ ]:
STUDENT_ID = ""    # your 學號, in quotes, e.g. "315707999"
PARTNER_ID = ""    # your partner's 學號

In [ ]:
URL  = ("https://raw.githubusercontent.com/202609-ML-FinTech/00-course-info/main/slides/"
        "default%20of%20credit%20card%20clients.csv")
CAND = ["default of credit card clients.csv", "../slides/default of credit card clients.csv",
        "../00-course-info/slides/default of credit card clients.csv"]
path = next((p for p in CAND if Path(p).exists()), URL)
credit = pd.read_csv(path).rename(columns={"X1":"LIMIT","X5":"AGE","X6":"PAY_0","X7":"PAY_2",
                                           "X12":"BILL_1","X18":"PAY_AMT_1","Y":"default"})
credit["LIMIT"] = credit["LIMIT"] / 10000          # NT$ ten-thousands
credit["BILL_1"] = credit["BILL_1"] / 10000
credit["PAY_AMT_1"] = credit["PAY_AMT_1"] / 10000
credit = credit[credit["PAY_0"] <= 8]                # one record has PAY_0 = 3032: a data error (IC-02). Dropped.
# 30,000 Taiwanese credit-card clients (2005). default = 1 if the client defaulted next month.
# PAY_0: months late on the most recent payment (-1 = paid on time, 0 = revolving, 1..9 months late).

def my_sample(student_id, n=3000):
    """Your desk's 3,000 applicants. The same 學號 always gives the same applicants."""
    seed = int(hashlib.sha256(f"{int(student_id)}-credit".encode()).hexdigest(), 16) % 2**32
    rng = np.random.default_rng(seed)
    return credit.iloc[np.sort(rng.choice(len(credit), n, replace=False))].copy()

assert str(STUDENT_ID).strip().isdigit(), "Fill in STUDENT_ID in the cell above, then run again."
desk = my_sample(STUDENT_ID)
print(f"Desk of {STUDENT_ID}: {len(desk)} applicants, {desk['default'].mean()*100:.1f}% defaulted")

---
# Step 1 — LDA by hand, one predictor (10 min)

Use PAY_0 (months late) alone, on all 30,000 clients:

| | mean of PAY_0 | prior π |
|---|---|---|
| paid (class 0) | μ₀ = -0.211 | 0.779 |
| defaulted (class 1) | μ₁ = 0.668 | 0.221 |

Pooled variance σ² = 1.130. The discriminant score is δ_k(x) = x·μ_k/σ² − μ_k²/(2σ²) + log π_k.

**Q1.** For an applicant with x = 1 (one month late), compute δ₀(1) = ____ and δ₁(1) = ____ . Which class does LDA pick?

**Q2.** Solve δ₀(x) = δ₁(x) for the boundary x*. Is it where you expected, between the two means? Why not halfway?

> _replace this line_

---
# Step 2 — Both models on your desk (12 min)

Split (given). Fit `LogisticRegression(max_iter=3000)` and `LinearDiscriminantAnalysis()` on
`train` with `LIMIT, PAY_0, PAY_2, AGE, BILL_1`. Print the test accuracy of each.

In [ ]:
from sklearn.model_selection import train_test_split
seed = int(hashlib.sha256(f"{int(STUDENT_ID)}-split".encode()).hexdigest(), 16) % 2**32
train, test = train_test_split(desk, test_size=0.4, random_state=seed, stratify=desk["default"])
print(len(train), "to fit,", len(test), "to test")

In [ ]:
# 2: both models; test accuracy



**Q3.** My desk: logistic accuracy ____ , LDA accuracy ____ .

**Q4.** The data team says "same accuracy, same model". What would you need to check before agreeing?

> _replace this line_

---
# Step 3 — Who is treated differently? (10 min)

On `test`, count the applicants where the two models **disagree** (one predicts default, the other
does not). Print the count and the first few such applicants (their LIMIT and PAY_0).

In [ ]:
# 3: disagreements



**Q5.** On my desk the models disagree on ____ of ____ test applicants. Describe one of them in words.

**Q6.** Swap with your partner.

| | logistic acc | LDA acc | disagreements |
|---|---|---|---|
| me | | | |
| my partner | | | |

Same accuracy, different people. Which model would you give the committee, and what would you tell them about the people in between?

> _replace this line_

---
# Step 4 — Hand in, on paper (5 min)

**One sheet per pair.** Write:

1. Both names and both 學號.
2. For **each** of you: both accuracies and the number of disagreements.
3. **Two sentences, agreed between you**, answering the client's question and the trap.

Give the sheet to Prof. Teng or Claire before you leave. **The sheet is the graded part.**

Then: *Restart & Run All*, and push this notebook as `in-class-exercise/1109/IC-1109.ipynb`
in your own repository before **12:10**.